##  Landelijke dataset maken

In [1]:

import pandas as pd

# =====================================
# 1. DATASET INLEZEN
# =====================================

df = pd.read_csv(
    "ongevallen.txt",
    sep=",",
    encoding="utf-8-sig",
    low_memory=False
)

print("Original dataset:", df.shape)


# =====================================
# 2. RELEVANTE KOLOMMEN SELECTEREN
# =====================================

columns = [
    "VKL_NUMMER",
    "DATUM_VKL",
    "JAAR_VKL",
    "MND_NUMMER",
    "GME_ID",
    "GME_NAAM",
    "WVK_ID",
    "JTE_ID",
    "MAXSNELHD",
    "AP3_CODE",
    "ANTL_DOD",
    "ANTL_GZH",
    "ANTL_SEH",
    "ANTL_SLA"
]

data = df[columns].copy()


# =====================================
# 3. DATA OPSCHONEN
# =====================================

# Numerieke kolommen
numeric_cols = [
    "JAAR_VKL",
    "MND_NUMMER",
    "MAXSNELHD",
    "ANTL_DOD",
    "ANTL_GZH",
    "ANTL_SEH",
    "ANTL_SLA"
]

for col in numeric_cols:
    data[col] = pd.to_numeric(
        data[col], errors="coerce"
    )

# Tekst opschonen
data["GME_NAAM"] = (
    data["GME_NAAM"].astype("string").str.strip()
)

data["AP3_CODE"] = (
    data["AP3_CODE"].astype("string").str.strip().str.upper()
)

data["WVK_ID"] = (
    data["WVK_ID"].astype("string").str.strip()
)

# Dubbele ongevallen verwijderen
data = data.drop_duplicates(subset="VKL_NUMMER")


# =====================================
# 4. JAREN FILTEREN
# =====================================

# Pas de jaren aan als jullie een andere periode willen.
data = data[
    data["JAAR_VKL"].between(2022, 2025)
].copy()


# =====================================
# 5. AMSTERDAM / REST NEDERLAND
# =====================================

data["area"] = "Rest Netherlands"

data.loc[
    data["GME_NAAM"].str.lower() == "amsterdam",
    "area"
] = "Amsterdam"


# =====================================
# 6. BEFORE / AFTER INTERVENTION
# =====================================

# Snelheidsverlaging: december 2023
# December 2023 behandelen we apart als overgangsperiode.

data["period"] = "Before"

data.loc[
    data["JAAR_VKL"] >= 2024,
    "period"
] = "After"

data.loc[
    (data["JAAR_VKL"] == 2023) &
    (data["MND_NUMMER"] == 12),
    "period"
] = "Transition"


# =====================================
# 7. NIEUWE KOLOM SPEED_CHANGED
# =====================================

# Nog onbekend: later vullen via koppeling WVK_ID.
# 1 = snelheid veranderd van 50 naar 30
# 0 = snelheid niet veranderd
# NA = nog niet geclassificeerd

data["speed_changed"] = pd.NA


# =====================================
# 8. DATASET OPSLAAN
# =====================================

data.to_csv(
    "ongevallen_gefilterd_landelijk.csv",
    index=False
)

print("Cleaned dataset:", data.shape)
print(data.head())


Original dataset: (1237603, 66)
Cleaned dataset: (508042, 17)
         VKL_NUMMER  DATUM_VKL  JAAR_VKL  MND_NUMMER  GME_ID        GME_NAAM  \
729561  20220000091        NaN      2022         NaN      74      Heerenveen   
729562  20220000139        NaN      2022         NaN    1731  Midden-Drenthe   
729563  20220000202        NaN      2022         NaN      47         Veendam   
729564  20220000203        NaN      2022         NaN    1690       De Wolden   
729565  20220000238        NaN      2022         NaN    1690       De Wolden   

             WVK_ID       JTE_ID  MAXSNELHD AP3_CODE  ANTL_DOD  ANTL_GZH  \
729561  386499077.0          NaN       50.0      UMS       NaN       NaN   
729562  468493002.0          NaN       60.0      UMS       NaN       NaN   
729563  512537007.0          NaN      100.0      UMS       NaN       NaN   
729564         <NA>  443439008.0       80.0      UMS       NaN       NaN   
729565  453429005.0          NaN       60.0      LET       NaN       NaN   



## Filteren naar alleen Amsterdam

In [2]:
import pandas as pd

# 1. Cleaned dataset inladen
df = pd.read_csv("ongevallen_gefilterd_landelijk.csv")

# 2. Alleen Amsterdam selecteren
amsterdam = df[
    df["GME_NAAM"].str.strip().str.lower() == "amsterdam"
].copy()

# 3. Opslaan als nieuw CSV-bestand
amsterdam.to_csv("ongevallen_gefilterd_amsterdam.csv", index=False)

# 4. Controleren
print("Total accidents:", len(df))
print("Accidents in Amsterdam:", len(amsterdam))

# Eerste 5 rijen bekijken
display(amsterdam.head())

Total accidents: 508042
Accidents in Amsterdam: 24963


,VKL_NUMMER,DATUM_VKL,JAAR_VKL,MND_NUMMER,GME_ID,GME_NAAM,WVK_ID,JTE_ID,MAXSNELHD,AP3_CODE,ANTL_DOD,ANTL_GZH,ANTL_SEH,ANTL_SLA,area,period,speed_changed
2182,20220012952,NaN,2022,NaN,363,Amsterdam,NaN,224377098.0,80.0,UMS,NaN,NaN,NaN,NaN,Amsterdam,Before,NaN
2205,20220013108,NaN,2022,NaN,363,Amsterdam,600122612.0,NaN,100.0,UMS,NaN,NaN,NaN,NaN,Amsterdam,Before,NaN
2584,20220014873,NaN,2022,NaN,363,Amsterdam,242377083.0,NaN,50.0,UMS,NaN,NaN,NaN,NaN,Amsterdam,Before,NaN
2586,20220014954,NaN,2022,NaN,363,Amsterdam,NaN,245381225.0,50.0,UMS,NaN,NaN,NaN,NaN,Amsterdam,Before,NaN
2587,20220015058,NaN,2022,NaN,363,Amsterdam,NaN,241370157.0,50.0,UMS,NaN,NaN,NaN,NaN,Amsterdam,Before,NaN


## Finetunen
Er is nu een kolom gemaakt die aangeeft of het ongeval plaatsvondt voor de verandering, na de verandering of in het transitie jaar (2023). De variablene zij 'before', 'after' en 'transition'. Omdat de maandcodes ontbreken in het originele bestand heb ik de transition variabele toegevoegd. Dus voor de duidelijkheid: transition is alles in het jaar 2023. 

In [3]:
import pandas as pd

amsterdam = pd.read_csv("ongevallen_gefilterd_amsterdam.csv")

# Kolom verwijderen
amsterdam = amsterdam.drop(columns=["area"])

# Bestand overschrijven
amsterdam.to_csv("ongevallen_gefilterd_amsterdam.csv", index=False)


In [6]:

import pandas as pd

# Dataset inladen
amsterdam = pd.read_csv("ongevallen_gefilterd_amsterdam.csv")

# Period opnieuw indelen
amsterdam["period"] = "Transition"

amsterdam.loc[
    amsterdam["JAAR_VKL"] <= 2022,
    "period"
] = "Before"

amsterdam.loc[
    amsterdam["JAAR_VKL"] >= 2024,
    "period"
] = "After"

# Controleer hoeveel ongevallen per periode
print(amsterdam["period"].value_counts())

# Controleer indeling per jaar
print(
    pd.crosstab(
        amsterdam["JAAR_VKL"],
        amsterdam["period"]
    )
)

# Dataset opslaan
amsterdam.to_csv(
    "ongevallen_gefilterd_amsterdam.csv",
    index=False
)


period
After         12171
Transition     6813
Before         5979
Name: count, dtype: int64
period    After  Before  Transition
JAAR_VKL                           
2022          0    5979           0
2023          0       0        6813
2024       6201       0           0
2025       5970       0           0
